# Phase 5: Full Experiments & Analysis

**Muc tieu:** Tao tat ca tables, figures, va analysis cho paper.

**Noi dung:**
1. Main Results Table (entity-level F1, Hits@1)
2. Traversal vs Selection Comparison
3. Ablation Studies
4. Cost vs Quality Analysis
5. Budget Sensitivity
6. Qualitative Analysis (Case Studies)
7. Error Analysis
8. Training Curves

**Evaluation protocol (matching GNN-RAG):**
- **Entity-level F1**: Predicted entities via eps=0.95 threshold vs gold entities
- **Hits@1**: top-1 predicted entity in gold answers?
- **Retrieval cost**: nodes visited during traversal
- **WebQSP dataset** (Freebase KG, single-hop + multi-hop)

**IMPORTANT notes:**
- Tat ca placeholder data deu la `?` - KHONG dien so gia truoc khi chay experiments
- GreedyGNN la baseline QUAN TRONG nhat: chung minh RL > greedy GNN score following
- So sanh Traversal vs Selection de chon approach tot nhat cho paper

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/GraphRAG_Research/utils')
from common import *

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 12})
import seaborn as sns

## 1. Load All Results

In [ ]:
# Load results tu cac phase truoc
results = {}

result_files = {
    'vector_rag': 'baseline_vector_rag_webqsp.json',
    'heuristic_bfs': 'baseline_heuristic_bfs_webqsp.json',
    'heuristic_beam': 'baseline_heuristic_beam_webqsp.json',
    'heuristic_ppr': 'baseline_heuristic_ppr_webqsp.json',
    'gnn_rag': 'gnn_rag_webqsp.json',
    'knowgpt': 'knowgpt_webqsp.json',
    'proposed_retrieval': 'proposed_retrieval_webqsp.json',
    'proposed_e2e': 'proposed_e2e_webqsp.json',
}

for key, filename in result_files.items():
    filepath = os.path.join(RESULTS_DIR, filename)
    if os.path.exists(filepath):
        with open(filepath) as f:
            results[key] = json.load(f)
        print(f'Loaded: {key}')
    else:
        print(f'Not found: {key} ({filepath})')

print(f'\nTotal results loaded: {len(results)} / {len(result_files)}')

## 2. Main Results Table (Table 1)

Metrics:
- **Hits@1**: top-1 predicted entity is correct?
- **F1**: entity-level F1 (eps=0.95 threshold)
- **Avg Cost**: average nodes visited per question

Note: GNN-RAG paper reports Hits@1 and F1 on WebQSP.
We add cost analysis (not in original GNN-RAG paper).

**KEY BASELINES:**
- **GreedyGNN**: Follow highest GNN scores without RL (critical for proving RL value)
- **GNN-RAG**: Published method - GNN scoring + shortest paths
- **GRL-RAG**: Our proposed method (best of Traversal vs Selection)

In [ ]:
# === TABLE 1: Main Results on WebQSP ===
# IMPORTANT: Tat ca so deu la '?' cho den khi chay xong experiments
# KHONG dien so gia (confirmation bias)

# GNN-RAG reported numbers (from paper, Table 2):
# GNN-RAG + ChatGPT: Hits@1 = 76.1
# Se so sanh voi GRL-RAG + ChatGPT

main_results = pd.DataFrame([
    {
        'Method': 'Vector RAG (Contriever)',
        'Type': 'Baseline',
        'Hits@1': '?', 'F1': '?',
        'Avg Cost': '?',
        'Notes': 'No graph structure'
    },
    {
        'Method': 'GraphRAG (BFS 2-hop)',
        'Type': 'Baseline',
        'Hits@1': '?', 'F1': '?',
        'Avg Cost': '?',
        'Notes': 'Fixed heuristic'
    },
    {
        'Method': 'GraphRAG (Beam w=5)',
        'Type': 'Baseline',
        'Hits@1': '?', 'F1': '?',
        'Avg Cost': '?',
        'Notes': 'Fixed heuristic'
    },
    {
        'Method': 'GraphRAG (PPR)',
        'Type': 'Baseline',
        'Hits@1': '?', 'F1': '?',
        'Avg Cost': '?',
        'Notes': 'Global ranking'
    },
    {
        'Method': 'GNN-RAG (ReaRev)',
        'Type': 'Reproduce',
        'Hits@1': '?', 'F1': '?',
        'Avg Cost': '?',
        'Notes': 'One-shot GNN scoring + shortest paths'
    },
    {
        'Method': 'GreedyGNN (ours)',
        'Type': 'Baseline',
        'Hits@1': '?', 'F1': '?',
        'Avg Cost': '?',
        'Notes': 'Follow top GNN scores greedily, no RL'
    },
    {
        'Method': 'GRL-RAG Traversal (ours)',
        'Type': 'Proposed',
        'Hits@1': '?', 'F1': '?',
        'Avg Cost': '?',
        'Notes': 'GNN-guided RL traversal + budget'
    },
    {
        'Method': 'GRL-RAG Selection (ours)',
        'Type': 'Proposed',
        'Hits@1': '?', 'F1': '?',
        'Avg Cost': '?',
        'Notes': 'RL path selection on GNN-RAG candidates'
    },
])

# Auto-fill from saved results (khi da chay xong)
method_to_file = {
    'GNN-RAG (ReaRev)': 'gnn_rag_webqsp.json',
    'GreedyGNN (ours)': 'greedy_gnn_webqsp.json',
    'GRL-RAG Traversal (ours)': 'grl_traversal_webqsp.json',
    'GRL-RAG Selection (ours)': 'grl_selection_webqsp.json',
}

for method_name, filename in method_to_file.items():
    filepath = os.path.join(RESULTS_DIR, filename)
    if os.path.exists(filepath):
        with open(filepath) as f:
            r = json.load(f)
        summary = r.get('summary', r)
        idx = main_results[main_results['Method'] == method_name].index[0]
        main_results.loc[idx, 'F1'] = f"{summary.get('avg_f1', '?'):.4f}"
        main_results.loc[idx, 'Hits@1'] = f"{summary.get('avg_hits', '?'):.4f}"
        main_results.loc[idx, 'Avg Cost'] = f"{summary.get('avg_cost', '?'):.1f}"
        print(f'Loaded results: {method_name}')

print('\n=== Table 1: Main Results on WebQSP ===')
print(main_results[['Method', 'Type', 'Hits@1', 'F1', 'Avg Cost']].to_markdown(index=False))
print('\nNote: Entity-level F1 (eps=0.95), matching GNN-RAG evaluation protocol')
print('CRITICAL: GreedyGNN must be LOWER than GRL-RAG to justify RL')
print('         If GreedyGNN ~= GRL-RAG => RL is unnecessary on local subgraphs')

## 2b. Traversal vs Selection Comparison (Table 1b)

**QUAN TRONG:** So sanh 2 approach de chon approach chinh cho paper.

| Approach | Description | Uu diem | Nhuoc diem |
|----------|-------------|---------|------------|
| **Traversal** | RL agent walk tren graph, chon neighbor tiep theo | Flexible, explore new paths | Co the cham, local subgraph nho |
| **Selection** | RL agent chon/loai paths tu GNN-RAG candidates | Fast, leverage GNN-RAG paths | Limited to GNN-RAG candidate set |
| **GreedyGNN** | Follow top GNN scores, no learning | Simple, no training | No learning, no context |

**Ket qua mong doi:**
- Neu Selection > Traversal: Subgraph qua nho cho RL traversal, GNN-RAG paths da du tot
- Neu Traversal > Selection: RL agent tim duoc paths ma GNN-RAG bo sot
- Neu GreedyGNN ~= cả hai: RL khong can thiet tren local subgraph => NEED TO RETHINK

In [ ]:
# === TABLE 1b: Traversal vs Selection vs GreedyGNN ===
# Focus comparison: 3 variants of our method

comparison_results = pd.DataFrame([
    {
        'Approach': 'GreedyGNN (no RL)',
        'Hits@1': '?', 'F1': '?',
        'Avg Cost': '?',
        'Training Time': 'N/A',
        'Description': 'Follow highest GNN scores greedily',
    },
    {
        'Approach': 'GRL-RAG Traversal',
        'Hits@1': '?', 'F1': '?',
        'Avg Cost': '?',
        'Training Time': '?',
        'Description': 'RL walk on local subgraph',
    },
    {
        'Approach': 'GRL-RAG Selection',
        'Hits@1': '?', 'F1': '?',
        'Avg Cost': '?',
        'Training Time': '?',
        'Description': 'RL select/filter GNN-RAG paths',
    },
])

# Auto-fill from saved results
approach_to_file = {
    'GreedyGNN (no RL)': 'greedy_gnn_webqsp.json',
    'GRL-RAG Traversal': 'grl_traversal_webqsp.json',
    'GRL-RAG Selection': 'grl_selection_webqsp.json',
}

for approach_name, filename in approach_to_file.items():
    filepath = os.path.join(RESULTS_DIR, filename)
    if os.path.exists(filepath):
        with open(filepath) as f:
            r = json.load(f)
        summary = r.get('summary', r)
        idx = comparison_results[comparison_results['Approach'] == approach_name].index[0]
        comparison_results.loc[idx, 'F1'] = f"{summary.get('avg_f1', '?'):.4f}"
        comparison_results.loc[idx, 'Hits@1'] = f"{summary.get('avg_hits', '?'):.4f}"
        comparison_results.loc[idx, 'Avg Cost'] = f"{summary.get('avg_cost', '?'):.1f}"
        print(f'Loaded: {approach_name}')

print('\n=== Table 1b: Traversal vs Selection ===')
print(comparison_results[['Approach', 'Hits@1', 'F1', 'Avg Cost', 'Training Time']].to_markdown(index=False))

print('\n--- Decision Guide ---')
print('1. If GreedyGNN >= GRL-RAG variants => RL NOT justified, need to rethink')
print('2. If Selection > Traversal => Use Selection as main method')
print('3. If Traversal > Selection => Use Traversal as main method')
print('4. Best variant becomes "GRL-RAG" in Table 1')

## 3. Ablation Studies (Table 2)

Remove each component to measure contribution:
1. **w/o GNN heuristic**: RL without GNN scores in state
2. **w/o RL (= GreedyGNN)**: Follow GNN scores greedily, no learning
3. **w/o Relation encoding**: Actions only encode entity, ignore relation type
4. **w/o Shared budget**: Reset budget per seed (not shared per question)
5. **w/o STOP action**: Force agent to use max_steps
6. **w/o Reward shaping**: REINFORCE without MINERVA-style potential-based shaping
7. **w/o IL pre-train**: RL from scratch (no imitation learning)

Note: Ablation applies to whichever approach (Traversal or Selection) performs best.

In [ ]:
# === ABLATION EXPERIMENT CONFIGS ===
# Updated to match P4a/P4b architecture

ablation_configs = [
    {
        'name': 'Full Model (GRL-RAG)',
        'use_gnn': True, 'use_rl': True, 'use_relation_enc': True,
        'shared_budget': True, 'use_stop': True,
        'use_shaping': True, 'use_pretrain': True,
    },
    {
        'name': 'w/o GNN heuristic',
        'use_gnn': False, 'use_rl': True, 'use_relation_enc': True,
        'shared_budget': True, 'use_stop': True,
        'use_shaping': False, 'use_pretrain': False,
        'description': 'Remove GNN scores from state + actions. No shaping (needs GNN).',
    },
    {
        'name': 'w/o RL (= GreedyGNN)',
        'use_gnn': True, 'use_rl': False, 'use_relation_enc': False,
        'shared_budget': True, 'use_stop': False,
        'use_shaping': False, 'use_pretrain': False,
        'description': 'Greedy: follow highest GNN-scored neighbors.',
    },
    {
        'name': 'w/o Relation encoding',
        'use_gnn': True, 'use_rl': True, 'use_relation_enc': False,
        'shared_budget': True, 'use_stop': True,
        'use_shaping': True, 'use_pretrain': True,
        'description': 'Actions encode only entity name, ignore relation type.',
    },
    {
        'name': 'w/o Shared budget',
        'use_gnn': True, 'use_rl': True, 'use_relation_enc': True,
        'shared_budget': False, 'use_stop': True,
        'use_shaping': True, 'use_pretrain': True,
        'description': 'Reset budget per seed (each seed gets full budget).',
    },
    {
        'name': 'w/o STOP action',
        'use_gnn': True, 'use_rl': True, 'use_relation_enc': True,
        'shared_budget': True, 'use_stop': False,
        'use_shaping': True, 'use_pretrain': True,
        'description': 'Force max_steps, no early stopping.',
    },
    {
        'name': 'w/o Reward shaping',
        'use_gnn': True, 'use_rl': True, 'use_relation_enc': True,
        'shared_budget': True, 'use_stop': True,
        'use_shaping': False, 'use_pretrain': True,
        'description': 'Sparse terminal reward only, no MINERVA-style shaping.',
    },
    {
        'name': 'w/o IL pre-train',
        'use_gnn': True, 'use_rl': True, 'use_relation_enc': True,
        'shared_budget': True, 'use_stop': True,
        'use_shaping': True, 'use_pretrain': False,
        'description': 'RL from scratch, no imitation learning warmup.',
    },
]

# All results start as '?'
ablation_results = pd.DataFrame([
    {'Variant': c['name'], 'Hits@1': '?', 'F1': '?', 'Avg Cost': '?'}
    for c in ablation_configs
])

# Auto-fill from saved results
for config in ablation_configs:
    safe_name = config['name'].lower().replace(' ', '_').replace('(', '').replace(')', '')
    filepath = os.path.join(RESULTS_DIR, f'ablation_{safe_name}.json')
    if os.path.exists(filepath):
        with open(filepath) as f:
            r = json.load(f)
        summary = r.get('summary', r)
        idx = ablation_results[ablation_results['Variant'] == config['name']].index[0]
        ablation_results.loc[idx, 'F1'] = f"{summary.get('avg_f1', '?'):.4f}"
        ablation_results.loc[idx, 'Hits@1'] = f"{summary.get('avg_hits', '?'):.4f}"
        ablation_results.loc[idx, 'Avg Cost'] = f"{summary.get('avg_cost', '?'):.1f}"
        print(f'Loaded ablation: {config["name"]}')

print('\n=== Table 2: Ablation Studies ===')
print(ablation_results.to_markdown(index=False))
print('\nRun each config variant in P4b and save results.')

## 4. Cost vs Quality Trade-off (Figure 1)

Scatter plot: Retrieval Cost (x) vs F1 (y) for all methods.
- Only plot methods that have actual results (skip '?' entries)
- Do NOT hardcode numbers - read from results files

In [ ]:
# === FIGURE 1: Cost vs Quality ===
# KHONG dung placeholder numbers - chi plot khi co actual results

# Define method styles (colors/markers)
method_styles = {
    'Vector RAG (Contriever)': {'color': '#9E9E9E', 'marker': 'o'},
    'GraphRAG (BFS 2-hop)': {'color': '#BDBDBD', 'marker': 's'},
    'GraphRAG (Beam w=5)': {'color': '#BDBDBD', 'marker': '^'},
    'GraphRAG (PPR)': {'color': '#BDBDBD', 'marker': 'D'},
    'GNN-RAG (ReaRev)': {'color': '#4CAF50', 'marker': '*'},
    'GreedyGNN (ours)': {'color': '#FF9800', 'marker': 'p'},
    'GRL-RAG Traversal (ours)': {'color': '#F44336', 'marker': '*'},
    'GRL-RAG Selection (ours)': {'color': '#2196F3', 'marker': '*'},
}

# Filter only methods with actual results
plot_data = main_results[
    (main_results['F1'] != '?') & (main_results['Avg Cost'] != '?')
].copy()
plot_data['F1_num'] = plot_data['F1'].astype(float)
plot_data['Cost_num'] = plot_data['Avg Cost'].astype(float)

if len(plot_data) == 0:
    print('No results available yet. Run experiments first!')
    print('This plot will auto-generate when result files are saved.')
else:
    fig, ax = plt.subplots(figsize=(10, 7))
    
    for _, row in plot_data.iterrows():
        name = row['Method']
        style = method_styles.get(name, {'color': 'gray', 'marker': 'o'})
        is_ours = 'ours' in name or 'GRL' in name
        size = 250 if is_ours else 120
        
        ax.scatter(row['Cost_num'], row['F1_num'],
                   s=size, c=style['color'], marker=style['marker'],
                   label=name, zorder=5, edgecolors='black', linewidth=0.5)
        
        offset_x = 8
        ax.annotate(name, (row['Cost_num'], row['F1_num']),
                    textcoords='offset points', xytext=(offset_x, 5),
                    fontsize=9, fontweight='bold' if is_ours else 'normal')
    
    ax.set_xlabel('Retrieval Cost (nodes expanded)', fontsize=14)
    ax.set_ylabel('Entity-level F1', fontsize=14)
    ax.set_title('Cost vs Quality Trade-off on WebQSP', fontsize=16)
    ax.set_xscale('log')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='lower right', fontsize=9, ncol=2)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'fig1_cost_vs_quality.pdf'),
                dpi=300, bbox_inches='tight')
    plt.show()
    print('Figure 1 saved.')

## 5. Budget Sensitivity (Figure 2)

How does performance change with different budget limits?
Shows that GRL-RAG agent learns to stop early when sufficient info collected.

In [ ]:
# === FIGURE 2: Budget Sensitivity ===
# Run GRL-RAG with different budget limits
# KHONG dung placeholder numbers

budgets = [5, 10, 20, 50, 100]

# Load results from files
budget_log_path = os.path.join(RESULTS_DIR, 'budget_sensitivity.json')

if os.path.exists(budget_log_path):
    with open(budget_log_path) as f:
        budget_data = json.load(f)
    
    budget_f1 = [budget_data[str(b)]['f1'] for b in budgets]
    budget_hits = [budget_data[str(b)]['hits'] for b in budgets]
    budget_actual_cost = [budget_data[str(b)]['avg_cost'] for b in budgets]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    # F1 vs Budget
    axes[0].plot(budgets, budget_f1, 'o-', color='#F44336', linewidth=2, markersize=8)
    axes[0].set_xlabel('Budget (max nodes)')
    axes[0].set_ylabel('Entity-level F1')
    axes[0].set_title('F1 vs Budget')
    axes[0].grid(True, alpha=0.3)

    # Hits@1 vs Budget
    axes[1].plot(budgets, budget_hits, 's-', color='#2196F3', linewidth=2, markersize=8)
    axes[1].set_xlabel('Budget (max nodes)')
    axes[1].set_ylabel('Hits@1')
    axes[1].set_title('Hits@1 vs Budget')
    axes[1].grid(True, alpha=0.3)

    # Actual Cost vs Budget
    axes[2].plot(budgets, budget_actual_cost, 'D-', color='#FF9800', linewidth=2, markersize=8)
    axes[2].plot(budgets, budgets, '--', color='gray', alpha=0.5, label='Budget = Cost')
    axes[2].set_xlabel('Budget (max nodes)')
    axes[2].set_ylabel('Actual Cost (nodes used)')
    axes[2].set_title('Actual Cost vs Budget')
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'fig2_budget_sensitivity.pdf'),
                dpi=300, bbox_inches='tight')
    plt.show()
    print('Figure 2 saved.')
else:
    print(f'Budget sensitivity results not found: {budget_log_path}')
    print('TODO: Run budget sensitivity experiments in P4b:')
    print('  for b in [5, 10, 20, 50, 100]:')
    print('    retriever.env.budget = b')
    print('    evaluate and save to budget_sensitivity.json')
    print()
    print('Expected insights:')
    print('  - F1 saturates at some budget (agent stops early)')
    print('  - Actual cost < budget (STOP action working)')
    print('  - Find optimal budget for cost/quality trade-off')

## 6. Comparison with GNN-RAG (Detailed)

Per-question analysis: when does GRL-RAG improve over GNN-RAG?

In [ ]:
# === Per-question comparison ===

def compare_methods(gnn_rag_results, proposed_results):
    """Compare GNN-RAG vs GRL-RAG per question."""
    gnn_by_qid = {r['qid']: r for r in gnn_rag_results}
    prop_by_qid = {r['qid']: r for r in proposed_results}
    
    common_qids = set(gnn_by_qid.keys()) & set(prop_by_qid.keys())
    
    wins = 0
    losses = 0
    ties = 0
    improvements = []
    
    for qid in common_qids:
        gnn_f1 = gnn_by_qid[qid].get('f1', 0)
        prop_f1 = prop_by_qid[qid].get('f1', 0)
        
        diff = prop_f1 - gnn_f1
        improvements.append(diff)
        
        if diff > 0.01:
            wins += 1
        elif diff < -0.01:
            losses += 1
        else:
            ties += 1
    
    print(f'Questions compared: {len(common_qids)}')
    print(f'GRL-RAG wins: {wins} ({wins/len(common_qids)*100:.1f}%)')
    print(f'GRL-RAG losses: {losses} ({losses/len(common_qids)*100:.1f}%)')
    print(f'Ties: {ties} ({ties/len(common_qids)*100:.1f}%)')
    print(f'Avg F1 improvement: {np.mean(improvements):.4f}')
    
    return improvements


# TODO: Run when results available
# improvements = compare_methods(
#     results['gnn_rag']['details'],
#     results['proposed_retrieval']['details']
# )

print('Per-question comparison function defined.')

## 7. Qualitative Analysis (Case Studies)

In [ ]:
# === CASE STUDIES ===
# Select questions that illustrate GRL-RAG's advantages

case_studies = [
    {
        'type': 'Multi-hop reasoning (GRL-RAG wins)',
        'question': 'TODO: Fill from actual results',
        'gold_answer': 'TODO',
        'gnn_rag': {
            'paths': 'TODO (shortest paths, may miss key relation)',
            'answer': 'TODO',
            'cost': 'TODO',
        },
        'grl_rag': {
            'paths': 'TODO (RL-selected path, follows content)',
            'answer': 'TODO',
            'cost': 'TODO',
        },
        'insight': 'RL agent follows informative relations instead of shortest distance',
    },
    {
        'type': 'Budget efficiency (early stopping)',
        'question': 'TODO',
        'gold_answer': 'TODO',
        'gnn_rag': {
            'cost': 'TODO (many paths, some noisy)',
        },
        'grl_rag': {
            'cost': 'TODO (stopped early, correct answer)',
        },
        'insight': 'STOP action prevents noise from irrelevant paths',
    },
    {
        'type': 'GNN heuristic guidance',
        'question': 'TODO',
        'gold_answer': 'TODO',
        'insight': 'GNN scores correctly rank answer entities, guiding RL to right region',
    },
]

for i, case in enumerate(case_studies):
    print(f'\n{"="*50}')
    print(f'Case {i+1}: {case["type"]}')
    print(f'{"="*50}')
    for k, v in case.items():
        if k != 'type':
            if isinstance(v, dict):
                print(f'  {k}:')
                for kk, vv in v.items():
                    print(f'    {kk}: {vv}')
            else:
                print(f'  {k}: {v}')

## 8. Error Analysis

In [ ]:
# === ERROR ANALYSIS ===
# Categorize failure cases of GRL-RAG

error_categories = {
    'entity_linking_fail': {
        'count': 0,
        'description': 'Seed entities not found in local graph',
        'fix': 'Better entity linking / fuzzy matching',
    },
    'graph_incomplete': {
        'count': 0,
        'description': 'Local subgraph does not contain answer path',
        'fix': 'Expand subgraph / use larger KG',
    },
    'wrong_path': {
        'count': 0,
        'description': 'RL agent chose wrong direction',
        'fix': 'More training / better reward signal',
    },
    'llm_generation_fail': {
        'count': 0,
        'description': 'Correct paths retrieved but LLM answered wrong',
        'fix': 'Larger LLM / better prompt',
    },
    'budget_insufficient': {
        'count': 0,
        'description': 'Answer requires more hops than budget allows',
        'fix': 'Increase budget for complex questions',
    },
}

# TODO: Analyze from actual failure cases
# for result in proposed_results:
#     if result['f1'] == 0:
#         category = classify_error(result)
#         error_categories[category]['count'] += 1

print('=== Error Categories ===')
for cat, info in error_categories.items():
    print(f'  {cat}: {info["count"]} ({info["description"]})')

## 9. Training Curves (Figure 3 / Appendix)

In [ ]:
# === TRAINING CURVES ===

train_log_path = os.path.join(RESULTS_DIR, 'rl_training_log.json')

if os.path.exists(train_log_path):
    with open(train_log_path) as f:
        train_log = json.load(f)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    
    window = 100  # Rolling average window
    
    # Reward
    rewards = pd.Series(train_log['rewards']).rolling(window).mean()
    axes[0].plot(rewards, color='#4CAF50')
    axes[0].set_title('Average Reward')
    axes[0].set_xlabel('Episode')
    axes[0].set_ylabel('Reward')
    axes[0].grid(True, alpha=0.3)
    
    # Loss
    losses = pd.Series(train_log['losses']).rolling(window).mean()
    axes[1].plot(losses, color='#F44336')
    axes[1].set_title('Policy Loss')
    axes[1].set_xlabel('Episode')
    axes[1].set_ylabel('Loss')
    axes[1].grid(True, alpha=0.3)
    
    # Cost
    costs = pd.Series(train_log['costs']).rolling(window).mean()
    axes[2].plot(costs, color='#2196F3')
    axes[2].set_title('Avg Retrieval Cost')
    axes[2].set_xlabel('Episode')
    axes[2].set_ylabel('Nodes Visited')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'fig3_training_curves.pdf'),
                dpi=300, bbox_inches='tight')
    plt.show()
    
    print('Figure 3 saved.')
else:
    print(f'Training log not found: {train_log_path}')
    print('Run P4b training first.')

## 10. Summary: All Figures & Tables for Paper

| Item | Content | Status |
|------|---------|--------|
| Table 1 | Main Results (entity-level F1, Hits@1, Cost) - includes GreedyGNN baseline | Template |
| Table 1b | Traversal vs Selection vs GreedyGNN comparison | Template |
| Table 2 | Ablation Studies (8 variants, includes relation encoding + shared budget) | Template |
| Table 3 | Case Studies (3 types) | Template |
| Figure 1 | Cost vs Quality scatter (all methods, auto from results) | Template |
| Figure 2 | Budget Sensitivity (F1, Hits@1, Cost vs budget, auto from results) | Template |
| Figure 3 | Training Curves (reward, loss, cost) | Template |
| Figure 4 | Architecture Diagram | Draw with draw.io |
| Figure 5 | Per-question comparison histogram | Template |

### KEY RESEARCH QUESTIONS to answer with experiments:
1. **Does RL improve over GreedyGNN?** (Table 1, Table 1b)
   - If NOT => RL is unnecessary on local subgraphs
2. **Which approach: Traversal or Selection?** (Table 1b)
   - If Selection wins => GNN-RAG paths sufficient, just need better filtering
   - If Traversal wins => RL finds new paths beyond GNN-RAG
3. **Which components matter most?** (Table 2)
   - Relation encoding? Shared budget? Reward shaping? IL pretrain?
4. **Cost efficiency?** (Figure 1, Figure 2)
   - Better F1 at lower cost than baselines?

### Paper structure suggestion:
1. Introduction + motivation (GNN one-shot vs RL sequential)
2. Related Work (GNN-RAG, KnowGPT, MINERVA)
3. Method (MDP formulation, GNN-guided policy, reward)
4. Experiments (Table 1, Table 2, Figure 1-2)
5. Analysis (Case studies, Error analysis)
6. Conclusion

### Target venues: EMNLP / NeurIPS
### Deadline: May 2026